# 01 · Data sanity check (SHE-23)

Run after `uv run python -m backend.load` and `uv run python -m backend.clean`.
Each section states what the README and data dictionary lead us to expect.

In [1]:
import json

from backend.paths import LOGS_DIR
from backend.warehouse import connect

con = connect()

## KEV additions by year

Expected: 2021: 311, 2022: 555, 2023: 187, 2024: 186, 2025: 245 (1,484 in total).

In [2]:
con.sql("""
    SELECT year(date_added) AS year, count(*) AS added
    FROM known_exploited
    GROUP BY ALL
    ORDER BY year
""").df()

,year,added
0,2021,311
1,2022,555
2,2023,187
3,2024,186
4,2025,245


## CVEs published per month, 2024 to 2026

92,205 were published in 2026 up to November 6.

In [3]:
con.sql("""
    SELECT strftime(published, '%Y-%m') AS month, count(*) AS published
    FROM vulnerabilities
    WHERE published >= DATE '2024-01-01'
    GROUP BY ALL
    ORDER BY month
""").df()

,month,published
0,2024-01,2594
1,2024-02,2778
2,2024-03,3309
3,2024-04,3621
4,2024-05,4999
5,2024-06,3078
6,2024-07,3113
7,2024-08,2896
8,2024-09,2514
9,2024-10,3564


## NIST review status of 2026 CVEs

Expected: 51,046 of 92,205 not yet analysed. Most of these have no product list,
which is why Granite reads the descriptions (SHE-7).

In [4]:
con.sql("""
    SELECT
        nvd_status,
        count(*) AS cves,
        nvd_status IN ('Received', 'Awaiting Analysis', 'Undergoing Analysis', 'Deferred')
            AS unreviewed
    FROM vulnerabilities
    WHERE year(published) = 2026
    GROUP BY ALL
    ORDER BY cves DESC
""").df()

,nvd_status,cves,unreviewed
0,Analyzed,35377,False
1,Deferred,35256,True
2,Awaiting Analysis,9714,True
3,Modified,5782,False
4,Received,5308,True
5,Undergoing Analysis,768,True


## KEV entries added before NIST published them

Expected: 62. Treat these as exploited at disclosure.

In [5]:
con.sql("""
    SELECT count(*) AS kev_before_published
    FROM known_exploited AS k
    JOIN vulnerabilities AS v USING (cve_id)
    WHERE k.date_added < v.published
""").df()

,kev_before_published
0,62


## Scoreboard snapshot months

June 2026 signals are missing, so CVEs published in June use July as their snapshot.
Expected: no `2026-06` snapshot, and all 7,921 June CVEs on `2026-07`.

In [6]:
con.sql("""
    SELECT
        strftime(published, '%Y-%m') AS published_month,
        snapshot_month,
        count(*) AS cves
    FROM scoreboard_2026
    GROUP BY ALL
    ORDER BY published_month
""").df()

,published_month,snapshot_month,cves
0,2026-01,2026-01,4295
1,2026-02,2026-02,4603
2,2026-03,2026-03,6227
3,2026-04,2026-04,5801
4,2026-05,2026-05,6922
5,2026-06,2026-07,7921
6,2026-07,2026-07,9757
7,2026-08,2026-08,12242
8,2026-09,2026-09,14902


## Cleaning results

Row counts from `uv run python -m backend.clean` (outputs/logs/clean_log.json).

In [7]:
json.loads((LOGS_DIR / "clean_log.json").read_text())

{'cwe': 441086,
 'signals_raw': 6463734,
 'signals_after_drop_rejected': 6312757,
 'signals_after_fill_feb_2025': 6327309,
 'signals_after_epss_pct': 6327309,
 'exploit_signals': 6327309,
 'vendor_aliases': 47,
 'kev': 1484,
 'affected_products': 3155934,
 'kev_vendor_matches': 1454}

## February 2025 gap

Expected: 14,552 February rows filled from January (`filled_gap = 1`), 267,661 February
rows in total.

In [8]:
con.sql("""
    SELECT month, filled_gap, count(*) AS rows
    FROM clean.exploit_signals
    WHERE month BETWEEN '2025-01' AND '2025-03'
    GROUP BY ALL
    ORDER BY month, filled_gap
""").df()

,month,filled_gap,rows
0,2025-01,0,262469
1,2025-02,0,253109
2,2025-02,1,14552
3,2025-03,0,272184
